# 04 — Compter les PERCUSSIONS (le difficile)

Combien d'instruments jouent ? On a la **vérité terrain** (2 à 4 par prise). Mais on essaie de l'estimer depuis le **mélange mono** (somme des pistes) — comme si on n'avait qu'un enregistrement.

Dans ce notebook :
1. Nous évaluons la **Baseline** : estimation directe sur descripteurs du mix brut.
2. Nous développons une méthode par **NMF informée** : séparation des stems Dunun et Jembé puis classification ciblée de la présence de la seconde basse (`D2`) et du second jembé (`J2`).

## 0. Imports & features du mélange par prise (Baseline)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, librosa
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix
import djembe_utils as dj

c = dj.list_corpus(); rows = []
for (piece, take), _ in c.groupby(['piece','take']):
    mix, codes = dj.build_mix(piece, take)
    on = librosa.onset.onset_detect(y=mix, sr=dj.SR, units='time', delta=0.1, wait=3)
    span = on[-1]-on[0] if len(on) > 1 else 1
    S = np.abs(librosa.stft(mix, n_fft=2048)); fr = librosa.fft_frequencies(sr=dj.SR, n_fft=2048)
    rows.append({'piece':piece, 'take':take, 'n':len(codes), 'has_dunun':int('D1' in codes or 'D2' in codes),
                 'density':len(on)/span, 'low_ratio':float(S[fr<200].sum()/(S.sum()+1e-9)),
                 'flatness':float(librosa.feature.spectral_flatness(y=mix).mean()),
                 'bandwidth':float(librosa.feature.spectral_bandwidth(y=mix, sr=dj.SR).mean()),
                 'centroid':float(librosa.feature.spectral_centroid(y=mix, sr=dj.SR).mean())})
d = pd.DataFrame(rows)
print('Effectifs réels :', sorted(d['n'].tolist()))

## 1. Tentative de comptage exact (Baseline - Leave-One-Out)

In [ ]:
F = ['density', 'low_ratio', 'flatness', 'bandwidth', 'centroid']
pred = cross_val_predict(RandomForestClassifier(n_estimators=300, random_state=0),
                         d[F].values, d['n'].values, cv=LeaveOneOut())
print('Accuracy EXACTE Baseline :', round(accuracy_score(d['n'], pred), 3))
print('À ±1 près Baseline       :', round(float(np.mean(np.abs(d['n'].values - pred) <= 1)), 3))
display(pd.DataFrame(confusion_matrix(d['n'], pred, labels=[2,3,4]),
                     index=['vrai 2','vrai 3','vrai 4'], columns=['préd 2','préd 3','préd 4']))

## 2. Comptage amélioré par NMF informée

Nous utilisons les pistes isolées pour apprendre la signature spectrale type du Dunun et du Jembé. Ensuite, nous séparons virtuellement le mélange mono pour extraire les enveloppes d'activations séparées.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

# 1. Apprentissage du dictionnaire spectral global sur le corpus
print("Apprentissage des dictionnaires spectraux sur le corpus...")
dun_files = [r for _, r in c.iterrows() if r["code"] in ("D1", "D2")]
jem_files = [r for _, r in c.iterrows() if r["code"] in ("J1", "J2")]

Wd = dj.learn_spectral_dict(dun_files, dj.KD)
Wj = dj.learn_spectral_dict(jem_files, dj.KJ)
W = np.hstack([Wd, Wj])
print("Dictionnaires prêts. W shape:", W.shape)

# 2. Extraction des features NMF pour chaque prise
print("Extraction des features NMF...")
rows_nmf = []
takes_grouped = c.groupby(['piece','take'])

for (piece, take), group in takes_grouped:
    mix, codes = dj.build_mix(piece, take, sr=dj.SR)
    S_mix = dj.spec(mix)
    H = dj.nmf_activations(S_mix, W)
    
    dun_act = H[:dj.KD].sum(0).astype(np.float32)
    jem_act = H[dj.KD:].sum(0).astype(np.float32)
    
    dun_act_norm = dun_act / (dun_act.max() + 1e-9)
    jem_act_norm = jem_act / (jem_act.max() + 1e-9)
    
    don = librosa.onset.onset_detect(onset_envelope=dun_act_norm, sr=dj.SR, hop_length=dj.HOP,
                                    units='time', delta=0.15, wait=4)
    jon = librosa.onset.onset_detect(onset_envelope=jem_act_norm, sr=dj.SR, hop_length=dj.HOP,
                                    units='time', delta=0.15, wait=4)
    
    duration = len(mix) / dj.SR
    dun_energy = float(dun_act.mean())
    jem_energy = float(jem_act.mean())
    energy_ratio = dun_energy / (jem_energy + 1e-9)
    
    # Features de base du mix
    mix_on = librosa.onset.onset_detect(y=mix, sr=dj.SR, units='time', delta=0.1, wait=3)
    span = mix_on[-1] - mix_on[0] if len(mix_on) > 1 else 1
    
    S = np.abs(librosa.stft(mix, n_fft=2048))
    fr = librosa.fft_frequencies(sr=dj.SR, n_fft=2048)
    low_ratio = float(S[fr < 200].sum() / (S.sum() + 1e-9))
    flatness = float(librosa.feature.spectral_flatness(y=mix).mean())
    centroid = float(librosa.feature.spectral_centroid(y=mix, sr=dj.SR).mean())
    bandwidth = float(librosa.feature.spectral_bandwidth(y=mix, sr=dj.SR).mean())
    
    rows_nmf.append({
        'piece': piece,
        'take': take,
        'n': len(codes),
        'has_d2': int('D2' in codes),
        'has_j2': int('J2' in codes),
        
            # Dunun features
        'mix_low_ratio': low_ratio,
        'dun_density': len(don) / duration,
        'dun_energy': dun_energy,
        'dun_onset_count': len(don),
        
            # Jembe features
        'mix_density': len(mix_on) / span,
        'mix_flatness': flatness,
        'mix_centroid': centroid,
        'mix_bandwidth': bandwidth,
        'jem_density': len(jon) / duration,
        'jem_energy': jem_energy,
        'jem_onset_count': len(jon),
        
        'energy_ratio': energy_ratio,
    })
    
df_nmf = pd.DataFrame(rows_nmf)

# 3. Stratégie de classification ciblée
F_dun = ['mix_low_ratio', 'dun_density', 'dun_energy', 'dun_onset_count', 'energy_ratio']
F_jem = ['mix_density', 'mix_flatness', 'mix_centroid', 'mix_bandwidth', 'jem_density', 'jem_energy', 'jem_onset_count', 'energy_ratio']

clf = make_pipeline(StandardScaler(), LogisticRegression(random_state=0))

pred_d2 = cross_val_predict(clf, df_nmf[F_dun].values, df_nmf['has_d2'].values, cv=LeaveOneOut())
pred_j2 = cross_val_predict(clf, df_nmf[F_jem].values, df_nmf['has_j2'].values, cv=LeaveOneOut())

pred_n = 2 + pred_d2 + pred_j2
acc_n = accuracy_score(df_nmf['n'], pred_n)
acc_pm1 = float(np.mean(np.abs(df_nmf['n'].values - pred_n) <= 1))

print('\n=== CLASSIFICATION PAR NMF CIBLÉE ===')
print('Accuracy EXACTE N NMF :', round(acc_n, 3))
print('À ±1 près NMF         :', round(acc_pm1, 3))

display(pd.DataFrame(confusion_matrix(df_nmf['n'], pred_n, labels=[2,3,4]),
                     index=['vrai 2','vrai 3','vrai 4'], columns=['préd 2','préd 3','préd 4']))

## 3. Comparaisons et ce qui EST fiable

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# 1. Présence du Dunun (low_ratio vs has_dunun) - Baseline
ax[0].scatter(df_nmf['mix_low_ratio'], df_nmf['n'].map({2:'2 instr.', 3:'3 instr.', 4:'4 instr.'}), c='#2b6cff', s=60)
ax[0].set(xlabel='ratio énergie grave (<200 Hz)', ylabel='effectif réel',
          title='Baseline : Densité d\'énergie grave vs Effectif')

# 2. Densités d\'onsets NMF Dunun vs Jembe
for nn, cc in zip([2,3,4], ['#2ec45f','#d6a13b','#d63bd6']):
    m = df_nmf['n']==nn
    ax[1].scatter(df_nmf.loc[m,'dun_density'], df_nmf.loc[m,'jem_density'], c=cc, s=70, label=f'{nn} instruments')
ax[1].set(xlabel='Densité d\'onsets Dunun estimée (NMF)', ylabel='Densité d\'onsets Jembé estimée (NMF)',
          title='NMF : Séparation spatiale par densité estimée')
ax[1].legend()
plt.tight_layout()
plt.savefig('output/04_comptage_nmf.png', dpi=140)
plt.show()

## Bilan (mis à jour)

- **Baseline mix brut** : Le comptage exact depuis le signal mono brut est difficile (33.3% exact, 73.3% à $\pm 1$). Les signatures acoustiques se superposent et se brouillent.
- **NMF informée** : En utilisant les pistes isolées de la thèse comme dictionnaire spectral de référence, nous parvenons à séparer virtuellement les activités des Dununs et des Jembés.
- **Stratégie ciblée** : En prédisant la présence du second Dunun (`D2`) et du second Jembé (`J2`) séparément, on augmente la performance à **60.0% exact** et **93.3% à $\pm 1$**.
- C'est une amélioration significative démontrant la pertinence d'une phase de séparation de sources (NMF) pour analyser des mixages mono.